In [1]:
import pandas as pd


df_watch_acc = pd.read_csv("../ignore_files/data/full/Watch_accelerometer.csv", index_col=0)
df_phone_acc = pd.read_csv("../ignore_files/data/full/Phones_accelerometer.csv", index_col=0)

df_watch_gyro = pd.read_csv("../ignore_files/data/full/Watch_gyroscope.csv", index_col=0)
df_phone_gyro = pd.read_csv("../ignore_files/data/full/Phones_gyroscope.csv", index_col=0)

# Concat watch and phone dataframes for accelerometer and gyroscope
df_acc = pd.concat([df_watch_acc, df_phone_acc], ignore_index=True)
df_gyro = pd.concat([df_watch_gyro, df_phone_gyro], ignore_index=True)

df_acc.info()
df_acc.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16603437 entries, 0 to 16603436
Data columns (total 9 columns):
 #   Column         Dtype  
---  ------         -----  
 0   Arrival_Time   int64  
 1   Creation_Time  int64  
 2   x              float64
 3   y              float64
 4   z              float64
 5   User           object 
 6   Model          object 
 7   Device         object 
 8   gt             object 
dtypes: float64(3), int64(2), object(4)
memory usage: 1.1+ GB


,Arrival_Time,Creation_Time,x,y,z,User,Model,Device,gt
0,1424696638740,27920678471000,-0.565032,-9.572019,-0.614113,a,gear,gear_1,stand
1,1424696638740,27920681910000,-0.832584,-9.713276,-0.606930,a,gear,gear_1,stand
2,1424696638740,27920692014000,-1.018134,-9.935339,-0.544082,a,gear,gear_1,stand
3,1424696638741,27920701983000,-1.222838,-10.142437,-0.566229,a,gear,gear_1,stand
4,1424696638741,27920711906000,-1.577180,-10.480618,-0.402824,a,gear,gear_1,stand


In [2]:
df_gyro.info()
df_gyro.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17138063 entries, 0 to 17138062
Data columns (total 9 columns):
 #   Column         Dtype  
---  ------         -----  
 0   Arrival_Time   int64  
 1   Creation_Time  int64  
 2   x              float64
 3   y              float64
 4   z              float64
 5   User           object 
 6   Model          object 
 7   Device         object 
 8   gt             object 
dtypes: float64(3), int64(2), object(4)
memory usage: 1.1+ GB


,Arrival_Time,Creation_Time,x,y,z,User,Model,Device,gt
0,1424696638743,27920678496000,-0.162187,-0.022104,0.059655,a,gear,gear_1,stand
1,1424696638743,27920681926000,-0.183225,-0.061785,0.012517,a,gear,gear_1,stand
2,1424696638743,27920692031000,-0.180829,-0.108657,-0.036485,a,gear,gear_1,stand
3,1424696638743,27920701997000,-0.147805,-0.157925,-0.098537,a,gear,gear_1,stand
4,1424696638744,27920743068000,0.182160,-0.323574,-0.277235,a,gear,gear_1,stand


In [3]:
# Grouped by matching permuations of (User, Model, Device, Arrival_Time, Creation_Time) between the gyroscope and accelerometer dataframes. This is to ensure that we are comparing the same data points between the two sensors.
# Save those that are not matching to a new dataframe for further inspection.
matching_columns = [
    "User",
    "Model",
    "Device",
    "Creation_Time",
]

merged_df = pd.merge(
    df_acc.assign(acc_index=df_acc.index),
    df_gyro.assign(gyro_index=df_gyro.index),
    on=matching_columns,
    how='inner'
)

unmerged_acc = df_acc[~df_acc.index.isin(merged_df['acc_index'])]
unmerged_gyro = df_gyro[~df_gyro.index.isin(merged_df['gyro_index'])]

# Lets change the column names to avoid confusion between the two sensors
column_names = {
    "X_x": "X_acc",
    "Y_x": "Y_acc",
    "Z_x": "Z_acc",
    "X_y": "X_gyro",
    "Y_y": "Y_gyro",
    "Z_y": "Z_gyro",
}

merged_df.rename(columns=column_names, inplace=True)

In [4]:
merged_df.to_csv('merged_data_oncreationtime.csv', index=False)
unmerged_acc.to_csv('unmerged_acc_oncreationtime.csv', index=False)
unmerged_gyro.to_csv('unmerged_gyro_oncreationtime.csv', index=False)
merged_df.head()


,Arrival_Time_x,Creation_Time,x_x,y_x,z_x,User,Model,Device,gt_x,acc_index,Arrival_Time_y,x_y,y_y,z_y,gt_y,gyro_index
0,1424696633911,207817520343346,-9.160782,-3.759674,1.396469,a,lgwatch,lgwatch_1,stand,42578,1424696633911,0.021973,0.003754,-0.071884,stand,42406
1,1424696633916,207817525287194,-9.198868,-3.788238,1.420273,a,lgwatch,lgwatch_1,stand,42579,1424696633916,0.025177,-0.000519,-0.071884,stand,42407
2,1424696633921,207817530231042,-9.208389,-3.804901,1.439316,a,lgwatch,lgwatch_1,stand,42580,1424696633921,0.030518,0.000549,-0.072952,stand,42408
3,1424696633926,207817535174889,-9.210770,-3.759674,1.446457,a,lgwatch,lgwatch_1,stand,42581,1424696633926,0.031586,0.002686,-0.069748,stand,42409
4,1424696633931,207817540118737,-9.222672,-3.735870,1.377426,a,lgwatch,lgwatch_1,stand,42582,1424696633931,0.033722,-0.000519,-0.067612,stand,42410


In [5]:
unmerged_acc.head()

,Arrival_Time,Creation_Time,x,y,z,User,Model,Device,gt
0,1424696638740,27920678471000,-0.565032,-9.572019,-0.614113,a,gear,gear_1,stand
1,1424696638740,27920681910000,-0.832584,-9.713276,-0.606930,a,gear,gear_1,stand
2,1424696638740,27920692014000,-1.018134,-9.935339,-0.544082,a,gear,gear_1,stand
3,1424696638741,27920701983000,-1.222838,-10.142437,-0.566229,a,gear,gear_1,stand
4,1424696638741,27920711906000,-1.577180,-10.480618,-0.402824,a,gear,gear_1,stand


In [6]:
unmerged_gyro.head()

,Arrival_Time,Creation_Time,x,y,z,User,Model,Device,gt
0,1424696638743,27920678496000,-0.162187,-0.022104,0.059655,a,gear,gear_1,stand
1,1424696638743,27920681926000,-0.183225,-0.061785,0.012517,a,gear,gear_1,stand
2,1424696638743,27920692031000,-0.180829,-0.108657,-0.036485,a,gear,gear_1,stand
3,1424696638743,27920701997000,-0.147805,-0.157925,-0.098537,a,gear,gear_1,stand
4,1424696638744,27920743068000,0.182160,-0.323574,-0.277235,a,gear,gear_1,stand


In [ ]:


acc_keys = df_acc[matching_columns].drop_duplicates()
gyro_keys = df_gyro[matching_columns].drop_duplicates()

matching_keys = acc_keys.merge(
    gyro_keys,
    on=matching_columns,
    how="outer",
    indicator=True,
)

acc_unmatched_keys = matching_keys.loc[
    matching_keys["_merge"].eq("left_only"), matching_columns
]
gyro_unmatched_keys = matching_keys.loc[
    matching_keys["_merge"].eq("right_only"), matching_columns
]

df_acc_not_matching = df_acc.merge(
    acc_unmatched_keys.assign(_acc_not_matching=True),
    on=matching_columns,
    how="inner",
).drop(columns="_acc_not_matching")
df_gyro_not_matching = df_gyro.merge(
    gyro_unmatched_keys.assign(_gyro_not_matching=True),
    on=matching_columns,
    how="inner",
).drop(columns="_gyro_not_matching")

df_acc_not_matching.insert(0, "Sensor", "accelerometer")
df_gyro_not_matching.insert(0, "Sensor", "gyroscope")
df_not_matching = pd.concat(
    [df_acc_not_matching, df_gyro_not_matching],
    ignore_index=True,
    sort=False,
)

df_not_matching.head()

df_not_matching.to_csv("sex.csv", index=False)
